<a href="https://colab.research.google.com/github/MohdSiddiq12/Regression_using_ANN/blob/main/CCPP_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Build an ANN Regression model to predict the electrical energy output of a Combined Cycle Power Plant

In [2]:
! pip install ucimlrepo
!pip install keras-tuner


In [15]:
import tensorflow as tf
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo
import keras_tuner as kt

In [5]:
combined_cycle_power_plant = fetch_ucirepo(id=294)

# data (as pandas dataframes)
X = combined_cycle_power_plant.data.features
y = combined_cycle_power_plant.data.targets

# metadata
# print(combined_cycle_power_plant.metadata)

# # variable information
# print(combined_cycle_power_plant.variables)


In [6]:
df = pd.concat([X, y], axis=1)

print(df.head())

      AT      V       AP     RH      PE
0  14.96  41.76  1024.07  73.17  463.26
1  25.18  62.96  1020.04  59.08  444.37
2   5.11  39.40  1012.16  92.14  488.56
3  20.86  57.32  1010.24  76.64  446.48
4  10.82  37.50  1009.23  96.62  473.90


In [7]:
#seperating values of dataset
X = df.iloc[:,:-1].values
Y = df.iloc[:,-1].values

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# Initialize Scaler
scaler = StandardScaler()

# Fit on training data AND transform it
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the SAME fitted scaler
X_test_scaled = scaler.transform(X_test)

The tf.keras.models.Sequential() function returns a new sequential model object that can be used to define
the architecture of a neural network. Once initialized, you can add layers to the model using the add() method.


In [16]:
def build_model(hp):
    model = tf.keras.models.Sequential()

    # Automatically tune the number of hidden layers (e.g., between 1 and 4)
    for i in range(hp.Int('num_layers', min_value=1, max_value=4)):
        model.add(
            tf.keras.layers.Dense(
                # Automatically tune the number of units/neurons per layer
                units=hp.Int(f'units_{i}', min_value=8, max_value=128, step=8),
                # Automatically test different activation functions
                activation=hp.Choice(f'activation_{i}', values=['relu', 'tanh'])
            )
        )

    # Output layer for regression (1 unit, no activation function)
    model.add(tf.keras.layers.Dense(units=1))

    # Automatically tune the learning rate for the Adam optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='mean_squared_error',
        metrics=['mean_absolute_error']
    )

    return model

In [17]:
tuner = kt.Hyperband(
    build_model,
    objective='val_loss',
    max_epochs=20,
    factor=3,
    directory='keras_tuner_dir',
    project_name='ccpp_tuning'
)
tuner.search(
    X_train_scaled,
    y_train,
    epochs=20,
    validation_data=(X_test_scaled, y_test)
)
best_model = tuner.get_best_models(num_models=1)[0]

Trial 30 Complete [00h 00m 23s]
val_loss: 17.565309524536133

Best val_loss So Far: 16.797618865966797
Total elapsed time: 00h 05m 47s


The mean squared error loss function is commonly used for regression problems. It computes the mean of the squared differences between the predicted and true values of the target variable. The goal of the training process is to minimize this loss function, which means that the model is trying to make its predictions as close as possible to the true values

In [18]:
history = best_model.fit(X_train_scaled, y_train, epochs=100, batch_size=32, validation_data=(X_test_scaled, y_test))

loss_df = pd.DataFrame(history.history)

# Add Epoch column starting at 1
loss_df.index = loss_df.index + 1
loss_df.index.name = 'Epoch'

# Display the loss values
print(loss_df)

Epoch 1/100
240/240 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 18.3178 - mean_absolute_error: 3.3199 - val_loss: 18.4678 - val_mean_absolute_error: 3.2960
Epoch 2/100
240/240 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 18.4355 - mean_absolute_error: 3.3252 - val_loss: 20.0764 - val_mean_absolute_error: 3.5339
Epoch 3/100
240/240 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 18.7356 - mean_absolute_error: 3.3569 - val_loss: 22.9946 - val_mean_absolute_error: 3.8140
Epoch 4/100
240/240 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 18.7002 - mean_absolute_error: 3.3512 - val_loss: 20.5898 - val_mean_absolute_error: 3.6000
Epoch 5/100
240/240 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 18.5721 - mean_absolute_error: 3.3498 - val_loss: 17.1752 - val_mean_absolute_error: 3.2526
Epoch 6/100
240/240 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 18.3797 - mean_absolute_error: 3.3233 - val_loss: 17.8122 - val_mean_absolute_error: 3.2625
Epoch 7/100
240/240 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 18.4118 - mean_absol